# Image Classification – Data Preparation & Label Check

This notebook prepares the image dataset before any model training. The data is **split first** (so no preprocessing decision can leak information from validation/test into training) and everything after that works on the partitioned folders:

* **Sections 1–2 Setup** – detect Colab / local, find the project folder, pick the compute device.
* **Section 3 Split & partition** – hash the raw images, quarantine duplicates in `train`, split the rest into train / val / test, copy the files into `data/{split}/`, then load the partitions and run sanity checks (missing files, label consistency, class balance per split).
* **Section 4 Quality check** – measure noise, blur and contrast per image; thresholds are chosen on the training split only.
* **Section 5 Restoration** – denoise / boost contrast / sharpen flagged images, with a strength that adapts to how bad each image is. Output goes to `data/{split}/images_corrected/`.
* **Section 6 Label audit with CLIP** – compare the labels to zero-shot CLIP predictions to find suspicious labels.
* **Section 7 Corrected labels** – overwrite the labels CLIP confidently disagrees with and save `labels_corrected.csv` per split.

Run the cells top to bottom.

## 1. Environment setup

The next cell checks `'google.colab' in sys.modules`:

* **Colab** – clones the GitHub repository into `/content/ImageClassification` (or `git pull`s if it is already there) and works from that folder.
* **Local** – uses the folder that contains `utils/` (the project root), so the notebook works no matter which folder Jupyter / VS Code was started in.

Either way all later paths are built from `ROOT`, so nothing is hard-coded.

In [5]:
import os, sys, subprocess
import warnings
warnings.filterwarnings('ignore')

REPO_URL = 'https://github.com/durak-dev/ImageClassification.git'
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Colab starts with an empty VM: clone the repo (code AND data live in it).
    # On later runs the folder already exists, so just pull the newest commits.
    ROOT = '/content/ImageClassification'
    if not os.path.exists(ROOT):
        subprocess.run(['git', 'clone', REPO_URL, ROOT], check=True)
    else:
        subprocess.run(['git', '-C', ROOT, 'pull'], check=True)
else:
    # Local: the notebook sits in the project root, but also work if it was
    # started from a subfolder.
    cwd = os.path.abspath(os.getcwd())
    ROOT = cwd if os.path.isdir(os.path.join(cwd, 'utils')) else os.path.dirname(cwd)

os.chdir(ROOT)
sys.path.insert(0, ROOT)          # so `import utils.funcs` finds OUR utils folder

DATA_DIR = os.path.join(ROOT, 'data')
ORIG_DIR = os.path.join(DATA_DIR, 'original')            # raw images/ + labels/ (placed manually, never modified)
MANIFEST_PATH = os.path.join(DATA_DIR, 'split_manifest.csv')   # which file belongs to which split
SEED = 42
print('Running on:', 'Google Colab' if IN_COLAB else 'local machine')
print('Project root:', ROOT)

Running on: Google Colab
Project root: /content/ImageClassification


## 2. Imports and hardware

`utils/funcs.py` holds all helper functions (plotting, quality metrics, restoration, CLIP helpers). It is reloaded on every run so edits show up immediately.
The device cell uses a CUDA GPU automatically when one is available (local NVIDIA GPU, or a Colab GPU runtime) and falls back to the CPU otherwise.

In [6]:
import importlib
import utils.funcs
importlib.reload(utils.funcs)      # pick up edits to funcs.py without restarting
from utils.funcs import *
from utils import config, github_sync
from utils.github_sync import save_csv, load_csv, push_files

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

In [7]:
DEVICE = get_device()              # 'cuda' if an NVIDIA GPU is usable, else 'cpu'
print('PyTorch device:', DEVICE, '-', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'no GPU found, using CPU')
print('TensorFlow GPUs:', tf.config.list_physical_devices('GPU'))
# Checkpoints: on Colab the VM disk disappears when the session ends, so (if GitHub
# a token is available) checkpoints are committed to GitHub instead of being
# kept on the VM. Locally they are plain files. Set USE_GITHUB manually to override.
if IN_COLAB:
    config.ensure_github()          # no .env / Colab Secret found -> asks for the token once (hidden input)
USE_GITHUB = IN_COLAB and config.github_ready()
_, _repo, _branch = config.github_settings()
print('Checkpoints ->', f'GitHub ({_repo}@{_branch})' if USE_GITHUB else 'local files in checkpoints/')
if IN_COLAB and not USE_GITHUB:
    print('Warning: no GitHub credentials found (.env / Colab Secrets) - checkpoints would be lost with the session')

PyTorch device: cuda - Tesla T4
TensorFlow GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Checkpoints -> local files in checkpoints/


## 3. Split and partition the data

All raw data lives in `data/original/` (put there manually, never modified):

```
data/original/images/   the raw images
data/original/labels.csv   the label table (columns: filename, class_id, description). It may also sit in a data/original/labels/ folder, which can additionally hold per-image label files with the same name as the image
```

The split happens **before** any quality check or restoration. `make_splits`:

1. hashes every image (SHA-256 of the file bytes) to find exact duplicates;
2. **quarantines** every image that has an identical twin: *all* copies go to `train`, none to `val` / `test`, so the same picture can never be on both sides of the split;
3. splits the remaining unique images in two steps, stratified by class: train vs. (val + test), then (val + test) into val and test. Val and test get 15% of *all* images each, so the final ratio is 70 / 15 / 15 even though the quarantined images are added to train.

The result is saved to `data/split_manifest.csv`. If that file exists it is **reused**, so everyone on the team gets exactly the same split (delete it to create a new one).

In [8]:
manifest = make_splits(ORIG_DIR, MANIFEST_PATH, val_frac=0.15, test_frac=0.15, seed=SEED)
manifest.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/ImageClassification/data/original/labels/labels.csv'

`materialize_splits` creates `data/{train,val,test}/{images,labels,images_corrected}/` and copies the raw files of each split into `images/` and `labels/` (the label table is cut to the rows of that split). `images_corrected/` stays empty until the restoration step. Files left over from an older split are removed, so a split can never contain another split's images.

In [ ]:
materialize_splits(ORIG_DIR, DATA_DIR, manifest)

**Leakage check.** Identical images must never appear in two splits, and duplicates may only live in `train`.

In [ ]:
assert manifest.groupby('hash').split.nunique().max() == 1, "an identical image appears in two splits"
assert (manifest[manifest.is_duplicate].split == 'train').all(), "duplicates must be quarantined in train"
assert manifest.filename.is_unique
print("OK - no identical image is shared between splits")
print(manifest.groupby('split').is_duplicate.agg(images='size', quarantined_duplicates='sum'))

### Load the partitions

`load_splits` reads `data/{split}/labels/labels.csv` for every split and adds a `path` column pointing into `data/{split}/images/` plus a `split` column. From here on, **all work uses these partitioned folders** – nothing reads `data/original/` any more.

In [ ]:
df = load_splits(DATA_DIR)
df.head(10)

**Check that every file in the CSV exists on disk.** A non-zero count usually means the repository was not fully cloned (or `data/` was not committed).

In [ ]:
missing = ~df.path.map(os.path.exists)
print(missing.sum(), "missing files")

**Label clean-up.** The class ids look like `n03042490` (WordNet ids); we turn them into integers. The next cell lists any id that maps to more than one description.

In [ ]:
df['class_id'] = df['class_id'].map(lambda x: int(x[1:]) if str(x).startswith('n') else x)

In [ ]:
unique_pairs = df[['class_id', 'description']].drop_duplicates()

conflicts_by_id = unique_pairs[unique_pairs.duplicated(subset=['class_id'], keep=False)].sort_values('class_id')

conflicts_by_desc = unique_pairs[unique_pairs.duplicated(subset=['description'], keep=False)].sort_values('description')

print("Conflicting ID mappings:")
print(conflicts_by_id)

### A first look at the data
A random sample of images with their descriptions, then how many images each class has.

In [ ]:
plot_images(df[df.split == 'train'], n=64, shuffle=True, seed=420, title_cols='description')

To assess class imbalance without overcrowding the axis with all 200 class names, we visualized the histogram using sampled tick labels. The discrete, step-like distribution reveals systematic capping, which could aid grokking.

In [ ]:
counts = plot_class_counts(df[df.split == 'train'], n_samples=16)

Class balance should be about the same in every split (the split is stratified). The table shows the number of images per split and how many classes each split contains.

In [ ]:
print(df.groupby('split').agg(images=('filename', 'size'), classes=('class_id', 'nunique')).loc[list(SPLITS)])
print("classes missing from train:", set(df.class_id) - set(df[df.split == 'train'].class_id))

## 4. Image quality check

`inspect_images` computes, on the grayscale image:

| metric | how | flag |
|---|---|---|
| **noise** | estimated noise **sigma** (Immerkaer filter), measured only in the flattest 20% of 8×8 tiles so that texture (fur, grass, foliage) is not mistaken for noise. ~1 = clean, 10+ = visibly grainy | `is_noisy` if `noise > NOISE_THRESHOLD` |
| **blur** | variance of the Laplacian – *lower means blurrier* | `is_blurry` if `blur < BLUR_THRESHOLD` |
| **contrast** | standard deviation of the pixel values – *lower means flatter / washed out* | `is_low_contrast` if `contrast < CONTRAST_THRESHOLD` |

The thresholds (5, 100 and 30) are starting points that are chosen by looking at the **training split only** (histogram and examples below), then applied unchanged to all splits; a contrast of 30 flags roughly the flattest 8% of images. The noise histogram typically shows a large clean cluster near 1 and a separate noisy tail – put the red line in the valley between them. Check the example grids and adjust if too many / too few images are flagged.
The grids show flagged images as they are and with a *fixed* correction strength, just to get a feeling for the filters.

In [ ]:
# 1. Measure quality and join the flags back onto df
NOISE_THRESHOLD, BLUR_THRESHOLD, CONTRAST_THRESHOLD = 5, 100, 30
quality = inspect_images(df, checks=['noise', 'blur', 'contrast'],
                         noise_threshold=NOISE_THRESHOLD, blur_threshold=BLUR_THRESHOLD,
                         contrast_threshold=CONTRAST_THRESHOLD)

quality_cols = [c for c in quality.columns if c != 'path']
df = df.drop(columns=quality_cols, errors='ignore')   # remove old results, if any
df = df.join(quality[quality_cols])

print(df[['is_noisy', 'is_blurry', 'is_low_contrast']].sum())
display(quality[df.split == 'train'].describe())   # use these numbers to pick better thresholds
plt.hist(df.noise[df.split == 'train'], bins=60); plt.axvline(NOISE_THRESHOLD, color='r')
plt.xlabel('estimated noise sigma'); plt.title('Noise distribution (red = threshold)'); plt.show()

# 2. Look at the flagged images: original vs a fixed-strength correction
noisy = df[(df.split == 'train') & df.is_noisy].sort_values('noise', ascending=False)
plot_images(noisy, n=16, title_cols='description')
plot_images(noisy, n=16, title_cols='description',
            transform=lambda img: denoise_image(img, strength=15))

blurry = df[(df.split == 'train') & df.is_blurry].sort_values('blur')
plot_images(blurry, n=16, title_cols='description')
plot_images(blurry, n=16, title_cols='description',
            transform=lambda img: deblur_image(img, amount=1.5))

## 5. Adaptive restoration

`fix_flagged_images` repairs every flagged image of a split and writes the result to `data/{split}/images_corrected/` (the raw images in `data/{split}/images/` are never modified; unflagged images are copied over unchanged, so `images_corrected/` is a complete copy of the split). Each image gets a strength based on its own scores:

* **Denoise** – the filter strength is tied directly to the measured noise: `h = 0.8 × sigma`, clipped to 3–15 (non-local means works best when `h` is close to the noise sigma). The result is then **blended** with the original, `out = α·denoised + (1−α)·original`, where α grows with severity (0.3 → 1). Mildly noisy images keep most of their detail, and a bad noise estimate cannot ruin an image. The NLM windows are shrunk automatically for small images (these are 64×64).
* **Contrast** – CLAHE (local histogram equalisation) on the lightness channel only, so colours stay put. Its clip limit goes from 1 to 3 with a log-scale severity of `contrast_threshold / contrast`, and the result is blended with the original by the same severity (≥ 0.3).
* **Sharpen** – amount 0.5–2 from a log-scale severity of `blur_threshold / blur` (the blur score spans several orders of magnitude), 25% weaker if the image is also noisy because noise inflates the blur score.
* **Order**: denoise → contrast → sharpen, so later steps do not amplify grain.

The table shows the strengths that were actually used. Below it, images are shown original vs fixed, first a random sample and then a spread across the noise range (mild → severe), which is the quickest way to see whether the strength matches the noise.

In [ ]:
# The thresholds are the ones chosen on the train split above. The correction is
# applied to every split separately: raw images from data/{split}/images/,
# results in data/{split}/images_corrected/.
fixed_parts = []
for split in SPLITS:
    fixed_parts.append(fix_flagged_images(
        df[df.split == split],
        out_folder=split_dir(DATA_DIR, split, 'images_corrected'),
        noise_threshold=NOISE_THRESHOLD, blur_threshold=BLUR_THRESHOLD,
        contrast_threshold=CONTRAST_THRESHOLD))
df_fixed = pd.concat(fixed_parts)

# Strength actually used per image (0 = that step was skipped)
display(df_fixed.loc[df_fixed.was_fixed, ['noise', 'denoise_strength', 'denoise_blend',
                                          'blur', 'deblur_amount', 'contrast', 'contrast_clip']].describe())

# compare a random sample before / after (same rows, original path vs fixed path)
changed = df_fixed[df_fixed.was_fixed]
plot_images(changed.assign(path=changed.orig_path), n=16, shuffle=True, seed=1, title_cols='description')   # original
plot_images(changed, n=16, shuffle=True, seed=1, title_cols='description')                                   # fixed

**Does the strength match the noise?** Same comparison, but 16 images spread evenly from the mildest to the noisiest flagged image. Mild ones should look almost untouched, severe ones clearly cleaner. If a texture-rich image looks smeared, lower `denoise_scale` or the upper end of `denoise_range`; if grainy images stay grainy, raise them.

In [ ]:
noisy_fixed = df_fixed[df_fixed.was_fixed & df_fixed.is_noisy].sort_values('noise')
spread = noisy_fixed.iloc[np.linspace(0, len(noisy_fixed) - 1, 16).astype(int)]
plot_images(spread.assign(path=spread.orig_path), n=16, title_cols='description')   # original, mild -> severe
plot_images(spread, n=16, title_cols='description')                                 # fixed

Images still flagged as blurry below are expected: sharpening cannot recover detail that was never captured. Note that `df_fixed` keeps the flags of the *original* images; re-run `inspect_images` on `df_fixed` to re-measure the fixed ones.

In [ ]:
df_fixed[df_fixed.is_blurry]

### Checkpoint

Saves / restores the `df_fixed` table (not the images). With `USE_GITHUB` the table is committed to the branch from `.env` (`GITHUB_BRANCH`, default `checkpoints`) and **no local copy is kept**; otherwise it is written to `checkpoints/df_fixed.csv`. Credentials are read by `utils/config.py` from `.env`, the environment or Colab Secrets – see `.env.example`.
Note: the table points at the files in `data/{split}/images_corrected/`; on Colab those images only exist after section 5 has run in the current session.

In [ ]:
save_csv(df_fixed, 'checkpoints/df_fixed.csv', ROOT, to_github=USE_GITHUB,
         message='checkpoint: df_fixed after restoration')

In [ ]:
df_fixed = load_csv('checkpoints/df_fixed.csv', ROOT, from_github=USE_GITHUB)

## 6. Label audit with CLIP

CLIP is only used as an independent judge, not trained. For each class we build the prompt *"a photo of a …"* and compare up to 20 images per class against all class prompts (cosine similarity), using the **corrected** images. The device chosen in section 2 decides whether this runs on the GPU.

Only the splits in `AUDIT_SPLITS` are audited and later relabelled. The `test` split is deliberately left out: if its labels were replaced by CLIP's opinion, test accuracy would partly measure agreement with CLIP instead of with the original ground truth.

In [ ]:
model, proc, dev = load_clip()

# One text prompt per class, e.g. "a photo of a barn"
classes = df_fixed.drop_duplicates('class_id')[['class_id', 'description']].reset_index(drop=True)
prompts = [f"a photo of a {first_name(d)}" for d in classes.description]
txt = clip_text_features(model, proc, dev, prompts)

# Up to 20 images per class (from the audited splits only), so this runs in reasonable time
AUDIT_SPLITS = ['train', 'val']
audited = df_fixed[df_fixed.split.isin(AUDIT_SPLITS)]
samp = audited.groupby('class_id', group_keys=False).apply(
    lambda g: g.sample(min(20, len(g)), random_state=69)
                 .assign(class_id=g.name)
)
cid2i = {c: i for i, c in enumerate(classes.class_id)}
y_true = samp.class_id.map(cid2i).values

img = clip_image_features(model, proc, dev, samp.path.values)
sims = img @ txt.T                                             # (N images, n classes) cosine similarities
y_pred = sims.argmax(1).numpy()
clip_score = (model.logit_scale.exp().item() * sims).softmax(1).max(1).values.numpy()
print("top 1:", np.mean(y_true == y_pred),
      "top 5:", (sims.topk(5).indices.numpy() == y_true[:, None]).any(1).mean())

`nbr_rank[i, j] == 1` means class *j* is the class closest in meaning to class *i*. A disagreement with a close neighbour (rank ≤ `K`) is probably CLIP confusing similar classes; a disagreement with an unrelated class is a **suspect label**.

In [ ]:
nbr_rank = class_neighbor_ranks(txt)   # nbr_rank[i, j] == 1: j is the class closest in meaning to i
K = 5

### Where does each class end up?
Confusion matrix between original and CLIP labels, plus a per-class summary.

In [ ]:
n = len(classes)
cmn = plot_confusion(y_true, y_pred, n)

# Per-class: where does each folder mostly go?
names = classes.description.map(first_name).values
top = cmn.argmax(1)
summary = pd.DataFrame({
    'original': names,
    'clip_top': names[top],
    'frac_top': cmn.max(1),
    'frac_self': np.diag(cmn),
})

### Suspect classes
Classes whose images mostly land on a *semantically unrelated* class.

In [ ]:
bad = summary[top != np.arange(n)].copy()          # indices, so two classes with the same first word can't collide
bad['nbr_rank'] = nbr_rank[bad.index, top[bad.index]]

suspect_cls = bad[bad.nbr_rank > K].sort_values('frac_top', ascending=False)
neighbor_cls = bad[bad.nbr_rank <= K].sort_values('frac_top', ascending=False)
print(f"{len(suspect_cls)} suspect classes, {len(neighbor_cls)} explained by semantic neighbours")
display(suspect_cls.head(40))

### Individual suspect images
Every sampled image is labelled `agree`, `neighbor` or `suspect`. The last cell shows suspect images where CLIP is confident (score > 0.6) – the best candidates for manual relabelling.

In [ ]:
res = pd.DataFrame({
    'path': samp.path.values,
    'filename': samp.filename.values,
    'split': samp.split.values,
    'pred_class_id': classes.class_id.values[y_pred],          # needed to write corrected labels
    'pred_description': classes.description.values[y_pred],
    'original': names[y_true],
    'clip_pred': names[y_pred],
    'clip_score': clip_score,
    'nbr_rank': nbr_rank[y_true, y_pred],
})
res['kind'] = np.where(y_true == y_pred, 'agree',
               np.where(res.nbr_rank <= K, 'neighbor', 'suspect'))
print(res.kind.value_counts())

save_csv(res, 'checkpoints/clip_results.csv', ROOT, to_github=USE_GITHUB,
         message='checkpoint: CLIP audit results')

In [ ]:
res = load_csv('checkpoints/clip_results.csv', ROOT, from_github=USE_GITHUB)  #Checkpoint: load results from disk to avoid re-running CLIP

In [ ]:
review = res[res.kind == 'suspect'].sort_values('nbr_rank', ascending=False)
display(review.head(30))      # most confident, semantically unrelated disagreements first

In [ ]:
thresshold = review[review.clip_score>0.6]
plot_images(thresshold, n=64, title_cols=['original', 'clip_pred', 'clip_score', 'nbr_rank'])

In [ ]:
thresshold = review[(review.clip_score>0.65) & (review.nbr_rank<15)]
plot_images(thresshold, n=64, title_cols=['original', 'clip_pred', 'clip_score', 'nbr_rank'])

## 7. Write the corrected labels

For every audited image where CLIP clearly disagrees with the original label, the original `class_id` and `description` are replaced by CLIP's prediction. An image is relabelled only if **all** of these hold:

* CLIP **disagrees** with the original label (without this check, images where CLIP agrees would pass the rank test, because a class is never its own neighbour);
* CLIP's confidence is **above** `MIN_CONF` (0.6);
* the predicted class is **not a close semantic neighbour** of the original: `nbr_rank >= MIN_NBR_RANK` (10). This separates a genuinely wrong label from CLIP confusing two similar classes.

Only images in the CLIP sample (up to 20 per class, `train` and `val`) were audited, so only those can be corrected; every other row keeps its original label. The decision uses the `res` table, so it also works after loading the `clip_results.csv` checkpoint (re-run the `res` cell once if the checkpoint predates the `pred_class_id` column).

In [ ]:
MIN_CONF, MIN_NBR_RANK = 0.6, 10

relabel = (res.kind != 'agree') & (res.clip_score > MIN_CONF) & (res.nbr_rank >= MIN_NBR_RANK)
changes = res[relabel].sort_values('clip_score', ascending=False).rename(
    columns={'original': 'was', 'pred_description': 'now'})
print(f"{len(changes)} of {len(res)} audited images will be relabelled")
print(changes.split.value_counts().to_string())
display(changes[['filename', 'split', 'was', 'now', 'clip_score', 'nbr_rank']].head(30))
plot_images(changes, n=32, title_cols=['was', 'now'])

The grid shows the relabelled images with the **original** label on top and the **new** label below. Skim it before saving: if many changes look wrong, raise `MIN_CONF` or `MIN_NBR_RANK` and re-run the previous cell.

In [ ]:
# One labels_corrected.csv per split, next to its labels.csv, in the same format
# (ids like n03042490). The class ids in `changes` are integers, so format them back.
new_id = dict(zip(changes.filename, changes.pred_class_id.map(lambda x: f"n{int(x):08d}")))
new_desc = dict(zip(changes.filename, changes.now))

for split in SPLITS:
    lab_dir = split_dir(DATA_DIR, split, 'labels')
    labels = pd.read_csv(os.path.join(lab_dir, LABELS_FILE))
    hit = labels.filename.isin(new_id)
    labels.loc[hit, 'class_id'] = labels.loc[hit, 'filename'].map(new_id)
    labels.loc[hit, 'description'] = labels.loc[hit, 'filename'].map(new_desc)
    out_path = os.path.join(lab_dir, CORRECTED_LABELS_FILE)
    labels.to_csv(out_path, index=False)
    print(f"{split:5s}: {hit.sum():5d} of {len(labels)} labels changed -> {out_path}")

`test` is not in `AUDIT_SPLITS`, so its `labels_corrected.csv` is an unchanged copy of `labels.csv` (this keeps the file layout identical for all splits). To train on the cleaned labels, load them with `load_splits(DATA_DIR, label_file=CORRECTED_LABELS_FILE, image_folder='images_corrected')`.

### Publish the results (Colab)

On Colab the split folders vanish with the session. If `USE_GITHUB` is on, this cell commits the split manifest and the three corrected label tables to GitHub in **one commit** (to the `GITHUB_BRANCH` from `.env`). Locally it does nothing – the files are already in `data/`.

In [ ]:
if USE_GITHUB:
    files = {'data/split_manifest.csv': MANIFEST_PATH}
    for split in SPLITS:
        files[f'checkpoints/labels_corrected_{split}.csv'] = os.path.join(
            split_dir(DATA_DIR, split, 'labels'), CORRECTED_LABELS_FILE)
    print(push_files(files, 'results: split manifest + corrected labels'))
else:
    print('Local run - nothing to upload')